Model

In [40]:
import numpy as np

class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, label=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.label = label


class CustomDecisionTree:
    def __init__(self, max_depth=10, min_samples=5, n_features=None, class_weights=None):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.n_features = n_features
        self.class_weights = class_weights
        self.root = None

    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.n_features_total = X.shape[1]
        self.n_features = self.n_features or self.n_features_total
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        n_samples = X.shape[0]

        # stopping conditions
        if (
            depth >= self.max_depth or
            n_samples < self.min_samples or
            len(np.unique(y)) == 1
        ):
            return DecisionTreeNode(label=self._majority_label(y))

        feature_idxs = np.random.choice(self.n_features_total, self.n_features, replace=False)

        best_feature, best_threshold = self._best_split(X, y, feature_idxs)

        if best_feature is None:
            return DecisionTreeNode(label=self._majority_label(y))

        left_mask = X[:, best_feature] < best_threshold
        right_mask = ~left_mask

        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return DecisionTreeNode(best_feature, best_threshold, left, right)

    def _best_split(self, X, y, feature_idxs):
        best_gini = float("inf")
        best_feature, best_threshold = None, None

        for feature in feature_idxs:
            X_col = X[:, feature]
            thresholds = np.unique(X_col)

            step = max(1, len(thresholds) // 10)

            for t in thresholds[::step]:
                gini = self._gini_split(y, X_col, t)

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = t

        return best_feature, best_threshold

    def _gini_split(self, y, X_col, threshold):
        left = y[X_col < threshold]
        right = y[X_col >= threshold]

        if len(left) == 0 or len(right) == 0:
            return float("inf")

        def weighted_gini(group):
            if len(group) == 0:
                return 0

            classes, counts = np.unique(group, return_counts=True)

            if self.class_weights is not None:
                weighted_counts = np.array([
                    counts[i] * self.class_weights.get(classes[i], 1)
                    for i in range(len(classes))
                ])
            else:
                weighted_counts = counts

            probs = weighted_counts / np.sum(weighted_counts)
            return 1 - np.sum(probs ** 2)

        n = len(y)
        return (len(left)/n)*weighted_gini(left) + (len(right)/n)*weighted_gini(right)

    def _majority_label(self, y):
        classes, counts = np.unique(y, return_counts=True)

        if self.class_weights is not None:
            weighted_counts = np.array([
                counts[i] * self.class_weights.get(classes[i], 1)
                for i in range(len(classes))
            ])
        else:
            weighted_counts = counts

        return classes[np.argmax(weighted_counts)]

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.label is not None:
            return node.label

        if x[node.feature] < node.threshold:
            return self._traverse(x, node.left)
        else:
            return self._traverse(x, node.right)

Training and Testing

In [ ]:
from preprocessing import preprocess 
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="pca",n_pca=50)  # balance=True for Phase 1, False for Phase 2

import time
from preprocessing import custom_classification_report, custom_confusion_matrix, accuracy_score

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(
    max_depth=10,
    min_samples=10,
    n_features=X_train.shape[1],          # adjust based on feature type
    class_weights=None  # use weights for Phase 1
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

# 4. Predict on validation set
print("\nEvaluating on Validation Data...")
val_preds = tree.predict(X_val)

# 5. Metrics (formatted like sklearn)
print("\n" + "="*45)
print("  CUSTOM MODEL VALIDATION PERFORMANCE")
print("="*45)

target_names = ["0 (Class 0)", "Not 0 (Class 1)"]  # Phase 1
print(classification_report(y_val, val_preds, target_names=target_names))

# Optional explicit accuracy
acc = accuracy_score(y_val, val_preds)
print(f"Accuracy: {acc:.4f}")

# 6. Confusion Matrix (labeled)
cm = confusion_matrix(y_val, val_preds)

print("\nConfusion Matrix (rows = actual, cols = predicted):\n")

labels = ["0", "1"]

# header
print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>8}", end="")
print()

# rows
for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:8}", end="")
    print()

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Initializing Custom Decision Tree...
Training the model...
Training completed in 8.76 seconds.

Evaluating on Validation Data...

  CUSTOM MODEL VALIDATION PERFORMANCE
                 precision    recall  f1-score   support

    0 (Class 0)       0.94      0.91      0.93       587
Not 0 (Class 1)       0.99      0.99      0.99      5413

       accuracy                           0.99      6000
      macro avg       0.97      0.95      0.96      6000
   weighted avg       0.99      0.99      0.99      6000

Accuracy: 0.9857

Confusion Matrix (rows = actual, cols = predicted):

                   0       1
         0      533      54
         1       32    5381


In [46]:
from preprocessing import preprocess 
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="hog",n_pca=50)  # balance=True for Phase 1, False for Phase 2

import time
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(
    max_depth=10,
    min_samples=10,
    n_features=X_train.shape[1],          # adjust based on feature type
    class_weights=None  # use weights for Phase 1
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

# 4. Predict on validation set
print("\nEvaluating on Validation Data...")
val_preds = tree.predict(X_val)

# 5. Metrics (formatted like sklearn)
print("\n" + "="*45)
print("  CUSTOM MODEL VALIDATION PERFORMANCE")
print("="*45)

target_names = ["0 (Class 0)", "Not 0 (Class 1)"]  # Phase 1
print(classification_report(y_val, val_preds, target_names=target_names))

# Optional explicit accuracy
acc = accuracy_score(y_val, val_preds)
print(f"Accuracy: {acc:.4f}")

# 6. Confusion Matrix (labeled)
cm = confusion_matrix(y_val, val_preds)

print("\nConfusion Matrix (rows = actual, cols = predicted):\n")

labels = ["0", "1"]

# header
print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>8}", end="")
print()

# rows
for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:8}", end="")
    print()

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Initializing Custom Decision Tree...
Training the model...
Training completed in 45.78 seconds.

Evaluating on Validation Data...

  CUSTOM MODEL VALIDATION PERFORMANCE
                 precision    recall  f1-score   support

    0 (Class 0)       0.92      0.91      0.92       587
Not 0 (Class 1)       0.99      0.99      0.99      5413

       accuracy                           0.98      6000
      macro avg       0.96      0.95      0.95      6000
   weighted avg       0.98      0.98      0.98      6000

Accuracy: 0.9837

Confusion Matrix (rows = actual, cols = predicted):

                   0       1
         0      533      54
         1       44    5369


In [45]:
from preprocessing import preprocess 
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="flatten",n_pca=50)  # balance=True for Phase 1, False for Phase 2

import time
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 2. Initialize model
print("Initializing Custom Decision Tree...")
tree = CustomDecisionTree(
    max_depth=10,
    min_samples=10,
    n_features=X_train.shape[1],          # adjust based on feature type
    class_weights=None  # use weights for Phase 1
)

# 3. Train model
print("Training the model...")
start_time = time.time()
tree.fit(X_train, y_train)
print(f"Training completed in {(time.time() - start_time):.2f} seconds.")

# 4. Predict on validation set
print("\nEvaluating on Validation Data...")
val_preds = tree.predict(X_val)

# 5. Metrics (formatted like sklearn)
print("\n" + "="*45)
print("  CUSTOM MODEL VALIDATION PERFORMANCE")
print("="*45)

target_names = ["0 (Class 0)", "Not 0 (Class 1)"]  # Phase 1
print(classification_report(y_val, val_preds, target_names=target_names))

# Optional explicit accuracy
acc = accuracy_score(y_val, val_preds)
print(f"Accuracy: {acc:.4f}")

# 6. Confusion Matrix (labeled)
cm = confusion_matrix(y_val, val_preds)

print("\nConfusion Matrix (rows = actual, cols = predicted):\n")

labels = ["0", "1"]

# header
print(f"{'':12}", end="")
for label in labels:
    print(f"{label:>8}", end="")
print()

# rows
for i, row in enumerate(cm):
    print(f"{labels[i]:>10} ", end="")
    for val in row:
        print(f"{val:8}", end="")
    print()

Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000
Initializing Custom Decision Tree...
Training the model...
Training completed in 74.71 seconds.

Evaluating on Validation Data...

  CUSTOM MODEL VALIDATION PERFORMANCE
                 precision    recall  f1-score   support

    0 (Class 0)       0.92      0.93      0.93       587
Not 0 (Class 1)       0.99      0.99      0.99      5413

       accuracy                           0.99      6000
      macro avg       0.96      0.96      0.96      6000
   weighted avg       0.99      0.99      0.99      6000

Accuracy: 0.9853

Confusion Matrix (rows = actual, cols = predicted):

                   0       1
         0      544      43
         1       45    5368
